# Preparação dos dados para modelagem — sem leakage

## Objetivo

A base `Bank Marketing` já contém o resultado observado da campanha, então usamos os rótulos reais em vez de gerar conversões sintéticas. Cada registro é preparado em três partes alinhadas:

- **contexto** (`X`): atributos disponíveis antes da decisão;
- **ação observada** (`contact`): canal histórico, `cellular` ou `telephone`;
- **recompensa observada** (`y`): `1` se houve assinatura do depósito, `0` caso contrário.

A base não possui uma data completa por linha; por isso a ordem original é preservada como proxy temporal. O histórico é observacional: não há atribuição aleatória nem recompensa contrafactual para o canal não utilizado. O pré-processamento é ajustado apenas com o treino para evitar que estatísticas ou categorias futuras influenciem as features.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Funciona na raiz do projeto, no diretório notebooks/ ou via nbconvert.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'src' / 'adaptive_offers').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Não encontrei a raiz do projeto (src/adaptive_offers).')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from adaptive_offers.config import DEFAULT_DATA_PATH, COLUMNS
from adaptive_offers.data import load_bank_marketing, validate_schema, sha256_file
from adaptive_offers.preparation import (
    ACTION_COLUMN, TARGET_COLUMN, LEAKAGE_COLUMNS, REPLACED_CONTEXT_COLUMNS,
    NUMERIC_CONTEXT_COLUMNS, CATEGORICAL_CONTEXT_COLUMNS,
    context_columns, derive_decision_features, modeling_frame,
    prepare_temporal_data, save_preparation_artifacts,
)

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / DEFAULT_DATA_PATH.name
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'{DATA_PATH} não existe. Baixe a base com `make data`.')

raw = load_bank_marketing(DATA_PATH)
contract = validate_schema(raw)
source_sha256 = sha256_file(DATA_PATH)
print(f'Fonte: {DATA_PATH.relative_to(PROJECT_ROOT)}')
print(f"Contrato: {contract['rows']:,} linhas × {contract['columns']} colunas | válido={contract['valid']}")
print(f'SHA-256: {source_sha256}')

## 1. Papéis das colunas

`contact` representa a ação usada historicamente e fica separada do contexto que existia antes da decisão. `y` é a recompensa observada após a campanha. `duration` é pós-decisão e não pode compor o contexto. `campaign` inclui a tentativa atual e `pdays=999` significa que nunca houve contato anterior; ambos recebem representações compatíveis com o instante da decisão.

In [ ]:
column_roles = pd.DataFrame([
    ('contexto candidato', 'age, job, marital, education, default, housing, loan', 'perfil/situação disponível antes da decisão'),
    ('contexto de campanha', 'month, day_of_week, previous, poutcome e indicadores macro', 'informação observável no momento da recomendação'),
    ('ação observada', 'contact', 'canal usado historicamente; fica fora de X'),
    ('recompensa/alvo', 'y', 'yes → 1 (assinou); no → 0'),
    ('pós-decisão / leakage', 'duration', 'só conhecida durante/depois do contato; excluída de X'),
    ('substituídos por atributos de decisão', 'campaign, pdays', 'incluem contato atual ou sentinela de nunca contatado'),
], columns=['papel', 'colunas', 'tratamento'])
display(column_roles)

## 2. Limpeza mínima e variáveis de decisão

A limpeza remove somente linhas exatamente duplicadas, mantendo a primeira ocorrência e a ordem original. Categorias `unknown` são valores explícitos da fonte e são preservadas. Ausências reais são verificadas pelo contrato e tratadas pelo imputador como salvaguarda. Registros com `y=no` permanecem: são resultados válidos, não dados a descartar.

A engenharia de decisão evita atribuir semântica incorreta às variáveis da campanha:

- `contacts_before = campaign - 1`, pois `campaign` já conta o contato atual;
- `never_contacted = 1` quando `pdays` é sentinela (`999` nesta cópia UCI; outra extração pode usar valor negativo);
- `pdays_contacted = 0` para quem nunca foi contatado e a distância real somente quando houve contato anterior.

O zero de `pdays_contacted` só é interpretável junto com `never_contacted`.

In [ ]:
clean = modeling_frame(raw)
dropped_duplicates = int(len(raw) - len(clean))
print(f'Linhas antes: {len(raw):,} | duplicatas removidas: {dropped_duplicates:,} | após limpeza: {len(clean):,}')
print(f"Respostas negativas mantidas como exemplos válidos: {(raw['y'] == 'no').sum():,}")
print("Categorias literais 'unknown' são mantidas como valores categóricos, não como ausências.")
assert clean.index.is_monotonic_increasing, 'A limpeza deve preservar a ordem de origem.'
assert not clean.duplicated().any(), 'Ainda há linhas duplicadas.'
assert clean.isna().sum().sum() == 0, 'O contrato esperado exige verificar ausências antes da preparação.'

engineered_preview = derive_decision_features(clean.head(8))
display(engineered_preview[[
    'campaign', 'contacts_before', 'pdays', 'never_contacted', 'pdays_contacted',
    'contact', 'y',
]])

## 3. Partições cronológicas

Usamos 60% para treino, 20% para validação e 20% para teste, na ordem da base e após remover duplicatas. Assim, registros posteriores não entram no treino. Como não há timestamp completo por linha, a posição no arquivo é uma aproximação temporal, não uma data individual verificada. As taxas e o mix de canais por partição servem para revelar mudanças no conjunto observado; não medem efeito causal do canal.

In [ ]:
prepared = prepare_temporal_data(raw, train_fraction=0.60, validation_fraction=0.20)
split_rows = []
for name, part in prepared.splits.as_dict().items():
    row = {
        'partição': name,
        'linhas': len(part),
        'primeiro índice de origem': part.index[0],
        'último índice de origem': part.index[-1],
        'conversão observada': (part[TARGET_COLUMN] == 'yes').mean(),
        'proporção celular observada': (part[ACTION_COLUMN] == 'cellular').mean(),
    }
    for action in ('cellular', 'telephone'):
        chosen = part[ACTION_COLUMN] == action
        row[f'n {action}'] = int(chosen.sum())
        row[f'conversão {action}'] = (part.loc[chosen, TARGET_COLUMN] == 'yes').mean()
    split_rows.append(row)
split_summary = pd.DataFrame(split_rows).set_index('partição')
display(split_summary.style.format({
    'conversão observada': '{:.2%}',
    'proporção celular observada': '{:.2%}',
    'conversão cellular': '{:.2%}',
    'conversão telephone': '{:.2%}',
}))

## 4. Contexto, ação e recompensa

O pipeline é ajustado **uma única vez no treino** e depois apenas transforma validação e teste. Categóricas usam imputação por `unknown` e one-hot encoding; numéricas usam mediana e escala calculadas no treino. Categorias não vistas são ignoradas pelo encoder, sem refazer o vocabulário com dados futuros.

`X_train`, `X_validation` e `X_test` contêm somente contexto. Os vetores `actions_*` e `rewards_*` mantêm, linha a linha, o canal e o resultado observados. Essa separação permite condicionar a estimativa de recompensa à ação sem tratar o canal histórico como característica do contexto da política.

In [ ]:
context_cols = list(prepared.context_columns)
excluded = sorted(set(COLUMNS) - set(context_cols))
feature_names = prepared.pipeline.get_feature_names_out().tolist()

print('Contexto bruto usado:')
print(context_cols)
print('Colunas excluídas/substituídas:')
print(excluded)
print(f'Features após transformação: {len(feature_names)}')
print('Exemplos de nomes transformados:', feature_names[:18])

matrix_summary = pd.DataFrame({
    name: {
        'linhas': matrix.shape[0],
        'features': matrix.shape[1],
        'valores finitos': bool(np.isfinite(matrix).all()),
        'ações alinhadas': len(prepared.actions[name]) == matrix.shape[0],
        'recompensas alinhadas': len(prepared.rewards[name]) == matrix.shape[0],
        'taxa de recompensa': prepared.rewards[name].mean(),
    }
    for name, matrix in prepared.matrices.items()
}).T
display(matrix_summary)

## 5. Verificações contra leakage

As asserções verificam conteúdo e alinhamento: ordem e separação cronológica; remoção ou substituição das colunas proibidas; recompensa binária derivada de `y`; ação preservada separadamente; matrizes finitas; estatísticas de imputação calculadas no treino; e categorias one-hot iguais às observadas no treino. A transformação de validação e teste não atualiza esses parâmetros.

In [ ]:
# Ordem das partições e retenção das linhas após deduplicação.
train, validation, test = (prepared.splits.train, prepared.splits.validation, prepared.splits.test)
assert train.index.max() < validation.index.min() < validation.index.max() < test.index.min()
assert sum(map(len, (train, validation, test))) == len(clean)
assert len(clean) == len(raw) - int(raw.duplicated().sum())

# Contexto e nomes pós-transformação não contêm ação, alvo, duração ou campos brutos substituídos.
forbidden = set(LEAKAGE_COLUMNS) | set(REPLACED_CONTEXT_COLUMNS)
assert forbidden.isdisjoint(context_cols)
assert forbidden.isdisjoint(feature_names)
assert {'contacts_before', 'never_contacted', 'pdays_contacted'}.issubset(context_cols)

# Matrizes, ação/recompensa e rótulos originais continuam linha a linha alinhados.
for name, part in prepared.splits.as_dict().items():
    assert prepared.matrices[name].shape[0] == len(part)
    assert np.isfinite(prepared.matrices[name]).all()
    assert np.array_equal(prepared.actions[name], part[ACTION_COLUMN].astype(str).to_numpy())
    expected_reward = (part[TARGET_COLUMN].astype(str).to_numpy() == 'yes').astype(np.int8)
    assert np.array_equal(prepared.rewards[name], expected_reward)
    assert set(np.unique(prepared.rewards[name])).issubset({0, 1})

# Medianas imputadas derivam do treino; categorias do encoder foram aprendidas no treino.
ct = prepared.pipeline.named_steps['context_preprocessor']
numeric_columns = list(ct.transformers_[0][2])
numeric_imputer = ct.named_transformers_['numeric'].named_steps['imputer']
train_numeric = derive_decision_features(train)
for column, fitted_median in zip(numeric_columns, numeric_imputer.statistics_):
    expected_median = train_numeric[column].median()
    assert np.isclose(fitted_median, expected_median), f'imputador não corresponde ao treino: {column}'

categorical_columns = list(ct.transformers_[1][2])
encoder = ct.named_transformers_['categorical'].named_steps['one_hot']
train_features = derive_decision_features(train)
for column, learned_categories in zip(categorical_columns, encoder.categories_):
    train_categories = set(train_features[column].fillna('unknown').astype(str).unique())
    assert set(map(str, learned_categories)) == train_categories, f'categorias fora do treino: {column}'

print('OK — split cronológico, sem duplicatas, sem leakage, vetores alinhados e estatísticas/categorias ajustadas no treino.')

## 6. Artefatos salvos

A preparação grava o pré-processador ajustado, as matrizes, os vetores de ações/recompensas e um manifesto com partições, colunas, proxies, duplicatas e período de ajuste. As transformações aprendidas ficam no artefato para aplicar a mesma codificação aos registros de validação e teste.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
manifest = save_preparation_artifacts(prepared, OUTPUT_DIR)
print(json.dumps(manifest, indent=2, ensure_ascii=False))
print(f'\nArtefatos gravados em: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}')
print('X_{train,validation,test}.npy: contexto transformado')
print('actions_{train,validation,test}.npy: canal histórico')
print('rewards_{train,validation,test}.npy: conversão binária')
print('context_preprocessor.joblib e preparation_manifest.json: transformação ajustada e proveniência')
assert manifest['fit_period'] == 'train only'
assert manifest['feature_count'] == prepared.matrices['train'].shape[1]
assert manifest['dropped_duplicates'] == int(raw.duplicated().sum())

## Resumo da preparação

A base original continua preservada em `data/raw/`; a transformação não inventa conversões nem descarta respostas negativas. O resultado é um contexto limpo e codificado, com ação histórica e recompensa binária alinhadas em partições cronológicas. A análise permanece limitada pela seleção histórica do canal, pela ausência de contrafactual e pelo uso da ordem de origem como proxy temporal.